In [ ]:
import os
import pandas as pd

from _plot_utils import plot_lift
from pass_pclr.defines import CINC_TARGETS


def load_experiment_data(runs_dir):
    experiments = {}

    # Get all experiment directories
    for exp_dir in sorted(os.listdir(runs_dir)):
        exp_path = os.path.join(runs_dir, exp_dir)

        # Check if it's a directory
        if os.path.isdir(exp_path):
            metrics_file = os.path.join(exp_path, "metrics.csv")
            probs_file = os.path.join(exp_path, "probs.npy")

            # Check if metrics.csv exists
            if os.path.exists(metrics_file):
                df = pd.read_csv(metrics_file)

                # Get AUROC values for multilabel tasks
                multilabel_aurocs = []
                multilabel_auprcs = []
                for task in CINC_TARGETS:
                    task_data = df[df["Label"] == task].iloc[0]
                    multilabel_aurocs.append(task_data["AUROC"])
                    multilabel_auprcs.append(task_data["AUPRC"])

                multilabel_avg_data = df[df["Label"] == "Multilabel Averaged"].iloc[0]

                experiments[exp_dir] = {
                    "multilabel_aurocs": multilabel_aurocs,
                    "multilabel_avg_auroc": multilabel_avg_data["AUROC"],
                    "multilabel_avg_auprc": multilabel_avg_data["AUPRC"],
                    "all_data": df,
                }

    return experiments

In [ ]:
experiments = {
    "full": load_experiment_data("../outputs/runs-cinc/"),
    "4k": load_experiment_data("../outputs/runs-cinc-4k/"),
    "2k": load_experiment_data("../outputs/runs-cinc-2k/"),
    "1k": load_experiment_data("../outputs/runs-cinc-1k/"),
    "512": load_experiment_data("../outputs/runs-cinc-512/"),
    "256": load_experiment_data("../outputs/runs-cinc-256/"),
}

print("Experiments found:")
for exp_subset, subset_results in experiments.items():
    print(f"\tSubset {exp_subset}:")
    for exp_name, exp_data in subset_results.items():
        print(f"\t\t{exp_name}: multilabel AUROC = {exp_data['multilabel_avg_auroc']:.4f}")

In [ ]:
sizes = {
    "full": 8192,
    "4k": 4096,
    "2k": 2048,
    "1k": 1024,
    "512": 512,
    "256": 256,
}

aliases = {
    # "proto-from-scratch": "proto-from-scratch",
    # "pass-heedb-pip": "pass-heedb-pip",
    # "pass-heedb-pit": "pass-heedb-pit",
    # "pass-heedb-pit-assign": "pass-heedb-pit-assign",
    # "prosup-heedb-pip": "prosup-heedb-pip",
    # "prosup-heedb-pit": "prosup-heedb-pit",
    # "prosup-heedb-pit-assign": "prosup-heedb-pit-assign",
    "proto-from-scratch-logreg": "proto-from-scratch-logreg",
    "pass-heedb-pip-logreg": "pass-heedb-pip-logreg",
    "pass-heedb-pit-logreg": "pass-heedb-pit-logreg",
    "pass-heedb-pit-assign-logreg": "pass-heedb-pit-assign-logreg",
    "prosup-heedb-pip-logreg": "prosup-heedb-pip-logreg",
    "prosup-heedb-pit-logreg": "prosup-heedb-pit-logreg",
    "prosup-heedb-pit-assign-logreg": "prosup-heedb-pit-assign-logreg",
}

palette = {
    # "proto-from-scratch": "tab:green",
    # "pass-heedb-pip": "tab:blue",
    # "pass-heedb-pit": "tab:orange",
    # "pass-heedb-pit-assign": "tab:purple",
    # "prosup-heedb-pip": "tab:cyan",
    # "prosup-heedb-pit": "tab:pink",
    # "prosup-heedb-pit-assign": "tab:olive",
    "proto-from-scratch-logreg": "tab:green",
    "pass-heedb-pip-logreg": "tab:blue",
    "pass-heedb-pit-logreg": "tab:orange",
    "pass-heedb-pit-assign-logreg": "tab:purple",
    "prosup-heedb-pip-logreg": "tab:cyan",
    "prosup-heedb-pit-logreg": "tab:pink",
    "prosup-heedb-pit-assign-logreg": "tab:olive",
}

df = pd.DataFrame.from_records(
    [
        {
            "Model": aliases[exp_name],
            "Train Size": sizes[exp_subset],
            "Multilabel (AUROC)": exp_data["multilabel_avg_auroc"],
            "Multilabel (AUPRC)": exp_data["multilabel_avg_auprc"],
        }
        for exp_subset, subset_results in experiments.items()
        for exp_name, exp_data in subset_results.items()
        if exp_name in aliases
    ]
)

In [ ]:
plot_lift(
    data=df,
    metric="Multilabel (AUROC)",
    palette=palette,
    title="CinC Georgia Multilabel (AUROC)",
    save_path="figs/cinc-multilabel-lift-roc.png",
    ylim=(0.5, 0.9),
    xlim=(256, 8192),
)
plot_lift(
    data=df,
    metric="Multilabel (AUPRC)",
    palette=palette,
    title="CinC Georgia Multilabel (AUPRC)",
    save_path="figs/cinc-multilabel-lift-pr.png",
    ylim=(0.05, 0.35),
    xlim=(256, 8192),
)